# 05 · Water vapour retrieval using original V6

**Starts the parallel WVD branch from the same seven inputs**, reading the raw CML, gauge and temperature files. The original V6 method is preserved (dry air ITU P.676, Graf CML-only wet screening, progressive baseline). For an immediate HPC demonstration reuse the existing V6 supporting NetCDF in the configured `extras.wvd_v6_nc`. For a NEW small retrieval set `RECOMPUTE_SMALL_V6=True` and select the number of E-band directional series. A 2021-only public download may yield differences from full historical calibration; the original V6 supporting product is NOT one of the seven public NetCDFs.

In [5]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [11]:
from essd_cz.connected import stage_wvd
RECOMPUTE_SMALL_V6=False        # True: run original V6 on small number of series (slow)
MAX_SERIES=3
PRESSURE_HPA=980.0
wvd_nc=stage_wvd(project,recompute=RECOMPUTE_SMALL_V6,
                 max_series=MAX_SERIES,pressure_hpa=PRESSURE_HPA)
print('Next WVD notebook will read:',wvd_nc)

[REGISTERED] wvd_v6: /home/jovyan/ESSD_publication_dataset/CZ_CML_WVD_EBAND_PROGRESSIVE_V6_2021.nc
  source: precomputed V6 supporting demonstration
Next WVD notebook will read: /home/jovyan/ESSD_publication_dataset/CZ_CML_WVD_EBAND_PROGRESSIVE_V6_2021.nc


In [12]:
import xarray as xr
with xr.open_dataset(project.get('wvd_v6')) as ds:
    print('V6 dimensions:',dict(ds.sizes))
    print('WVD variables (sample):',list(ds.data_vars)[:24])

V6 dimensions: {'series': 1352, 'time': 61632, 'method': 2}
WVD variables (sample): ['cml_id', 'sublink_id', 'frequency_ghz', 'length_km', 'polarization', 'site_a_latitude', 'site_a_longitude', 'site_b_latitude', 'site_b_longitude', 'gauge_distance_km', 'temperature_distance_km', 'temperature_ref_c', 'p676_sensitivity_dbkm_per_gm3', 'gauge_station_id', 'temperature_station_id', 'temperature_ref_n', 'temperature_5min_c', 'p676_monotonic', 'graf_q80_db', 'graf_threshold_db', 'baseline_B_db', 'baseline_pre_db', 'baseline_post_db', 'baseline_delta_post_minus_pre_db']
